# Example — Funnel

Journey example 1 — Funnel.

Companion script: `01_funnel_example.py`

**Run with cwd = repo root** (or the script's folder / `examples`).


## Setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

HERE = Path.cwd()
if (HERE / "day-1").exists():
    _DAY1_ROOT = HERE / "day-1"
elif HERE.name in {"01_sessionize", "02_journey", "03_state_modeling", "04_feature_engineering", "05_segmentation", "_shared"}:
    _DAY1_ROOT = HERE.parent
elif HERE.name == "examples":
    _DAY1_ROOT = HERE.parents[1]
else:
    _DAY1_ROOT = HERE

# Make Path(__file__) match the companion script (same as running the .py).
__file__ = str(_DAY1_ROOT / "02_journey/examples/01_funnel_example.py")
print("DAY1 root =", _DAY1_ROOT)
print("__file__ =", __file__)


## Imports & constants


In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
import sys
_DAY1 = Path(__file__).resolve().parents[2]
_SHARED = _DAY1 / "_shared"
if str(_SHARED) not in sys.path:
    sys.path.insert(0, str(_SHARED))
from toy_sessions import FUNNEL_STAGES, describe_sessions, load_toy_events


## Helpers (one function per cell)


### `funnel_table`


In [ ]:
def funnel_table(events: pd.DataFrame) -> pd.DataFrame:
    """Same idea as step02_journey_analysis.funnel_table (toy-sized)."""
    event_counts = events.groupby("stage", sort=False).size()
    presence = events.groupby(["session_idx", "stage"], sort=False).size().unstack(fill_value=0)
    for stage in FUNNEL_STAGES:
        if stage not in presence.columns:
            presence[stage] = 0
    contains = presence[FUNNEL_STAGES].gt(0)

    rank = {stage: index for index, stage in enumerate(FUNNEL_STAGES)}
    furthest = events["stage"].map(rank).groupby(events["session_idx"], sort=False).max()

    rows = []
    for index, stage in enumerate(FUNNEL_STAGES):
        containing = int(contains[stage].sum())
        next_stage = FUNNEL_STAGES[index + 1] if index + 1 < len(FUNNEL_STAGES) else ""
        advanced = int((contains[stage] & contains[next_stage]).sum()) if next_stage else 0
        rows.append(
            {
                "stage": stage,
                "events": int(event_counts.get(stage, 0)),
                "sessions_containing_stage": containing,
                "sessions_ending_here": int((furthest == index).sum()),
                "next_stage": next_stage or None,
                "advance_rate": round(advanced / containing, 4) if next_stage and containing else np.nan,
            }
        )
    return pd.DataFrame(rows)


### `_section`


In [ ]:
def _section(title: str) -> None:
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


### `main`


In [ ]:
def main() -> None:
    _section("FUNNEL — reach & advance (stage tags)")
    print("Goal: raw staged events → funnel reach / advance table.")
    describe_sessions()
    events = load_toy_events()

    _section("RAW — input events with stages")
    print(events[["session_idx", "anonymous_id", "timestamp", "event_name", "stage"]].to_string(index=False))

    table = funnel_table(events)
    _section("PROCESSED — funnel table")
    print(table.to_string(index=False))
    print("\nNote: advance_rate = share that also hit the NEXT stage (ever), not next click.")
    print("  landing→scroll = 0.75; intent→deep = 0.50 (cliff).")


## Run


In [ ]:
main()
